# Sanad سند — question answering over Saudi law

## Part 10: answers that show their text

The fine-tuned models answer in a few words ("the court [1]"), because the training targets were the short ArabLegalEval answers. Here each target is the short answer followed by the sentence of the cited article it rests on, picked automatically as the sentence sharing the most words with the answer and question: «…، إذ ينص النص [1] على: «…»». Same rewrites and 12% "not found" examples as part 4, so the only change is the form of the answer.

**Input:** the output of part 4 (the rewrites and the 12% adapter, for comparison). **Settings:** Internet on, GPU accelerator.

Code: https://github.com/readaloush/sanad

In [ ]:
!pip -q install sentence-transformers rank_bm25 peft bitsandbytes accelerate
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import glob
REWRITES = glob.glob('/kaggle/input/**/rewrites.json', recursive=True)[0]
LORA2 = glob.glob('/kaggle/input/**/lora2/adapter_config.json', recursive=True)[0].rsplit('/', 1)[0]
print(REWRITES, LORA2)

In [ ]:
!python -m sanad.sft --model ALLaM-AI/ALLaM-7B-Instruct-preview --rewrites {REWRITES} --p-unanswerable 0.12 --full-answers --data data --out results/lora4 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!python -m sanad.answer_eval --models ALLaM-AI/ALLaM-7B-Instruct-preview --adapter results/lora4 --tag _lora4 --skip-mcq --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!python -m sanad.demo_answers --search rerank --adapters lora2={LORA2} lora4=results/lora4 --data data --out results/demo.json 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!mkdir -p /kaggle/working/out && cp results/answers_lora4.json results/answer_samples_lora4.json results/demo.json /kaggle/working/out/ && cp -r results/lora4 /kaggle/working/out/lora4
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out